# 02 · Thiết kế mẫu và batch effect

Công việc 1.3 (TV1). Sản phẩm: bảng chéo thiết kế mẫu, hình F1, phân tích confounding, PCA + kiểm định batch effect (H6), `config/experiment_config.yaml`. Tóm tắt cho báo cáo: `docs/report/1.5_thiet_ke_thi_nghiem.md`.

In [ ]:
import os
from pathlib import Path

# Chuyển về thư mục gốc dự án để import được src/
while not (Path.cwd() / "src").is_dir():
    os.chdir("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("default")
from src.config import PALETTE, CURATED_DIR, REPORT_FIG_DIR, SLIDE_FIG_DIR, METRICS_DIR
from src.load import load_golub

X, samples = load_golub("curated")
MARKERS = {"ALL": "o", "AML": "^"}   # ALL hình tròn, AML tam giác (style guide)

print("Thư mục làm việc:", Path.cwd())
print("X:", X.shape, "| samples:", samples.shape)

## 1. Bảng chéo thiết kế mẫu: Đếm số mẫu theo tập (split), loại mô (tissue), nguồn (source) và lớp (class).

In [ ]:
samples.head()

In [ ]:
# đếm số mẫu cho từng tổ hợp giá trị
pd.crosstab([samples["split"], samples["tissue"]], samples["class"])

In [ ]:
pd.crosstab([samples["split"], samples["tissue"]], samples["class"],
            margins=True, margins_name="Tổng")

In [ ]:
pd.crosstab([samples["split"], samples["class"]],
            [samples["source"], samples["tissue"]])

**Nhận xét:**
1. **Tập train**: Toàn bộ 27 bệnh nhân ALL đến từ DFCI, toàn bộ 11 bệnh nhân AML đến từ CALGB. Không nguồn nào cung cấp cả hai loại bệnh. St-Jude và CCG chỉ xuất hiện trong tập test.
2. **Loại mô**: Tập train không có mẫu máu ngoại vi (PB), PB chỉ xuất hiện trong tập test (10 mẫu, từ DFCI, CALGB và CCG).
3. **Tập test**: St-Jude là nguồn duy nhất cung cấp cả ALL (3) và AML (5), tất cả là mẫu BM. Vì trong St-Jude nguồn mẫu không đổi, mô hình chỉ phân loại đúng các mẫu này nếu dựa vào tín hiệu bệnh chứ không phải tín hiệu bệnh viện. St-Jude vì vậy là phép kiểm tra quan trọng cho nghi ngờ "học bệnh viện", dù số mẫu nhỏ (n = 8).
4. **Hệ quả**: Trong tập train, lớp bệnh bị gây nhiễu hoàn toàn (confounding) với nguồn mẫu: ALL ↔ DFCI, AML ↔ CALGB. Vì vậy, mọi khác biệt giữa ALL và AML trên tập train có thể đến từ bệnh, nhưng cũng có thể đến từ khác biệt kỹ thuật giữa hai nguồn (quy trình lấy mẫu, bảo quản, xử lý chip — gọi là batch effect), và dữ liệu train không đủ để tách hai nguyên nhân này. Một mô hình học trên train hoàn toàn có thể "ăn gian" bằng cách nhận ra nguồn mẫu thay vì nhận ra bệnh, mà độ chính xác trên train vẫn cao. Đây mới là nguy cơ, chưa phải kết luận.

## 2. Hình F1: thiết kế mẫu

Biểu đồ cột ngang số mẫu theo nguồn × loại mô × lớp, tách hai panel train/test.

In [ ]:
def ve_F1(samples, co_chu=10, kich_thuoc=(10, 4.5)):
    """Vẽ hình F1: số mẫu theo nguồn × loại mô × lớp, hai panel train/test."""
    # (A) Danh sách các nhóm (nguồn, loại mô) có trong dữ liệu -> các hàng trên trục y
    nhom = samples[["source", "tissue"]].drop_duplicates().sort_values(["source", "tissue"])
    nhan_truc_y = [f"{s} – {t}" for s, t in zip(nhom["source"], nhom["tissue"])]
    vi_tri = np.arange(len(nhom))
    cao = 0.4

    # (B) Tạo khung hình gồm 2 panel cạnh nhau
    fig, axes = plt.subplots(1, 2, figsize=kich_thuoc, sharex=True, sharey=True)

    # (C) Vẽ lần lượt panel train rồi panel test
    for ax, tap in zip(axes, ["train", "test"]):
        con = samples[samples["split"] == tap]
        dem = (pd.crosstab([con["source"], con["tissue"]], con["class"])
                 .reindex(pd.MultiIndex.from_frame(nhom), fill_value=0)
                 .reindex(columns=["ALL", "AML"], fill_value=0))

        # (D) Mỗi nhóm vẽ 2 thanh: ALL (xanh, trơn) và AML (cam, gạch chéo)
        for i, lop in enumerate(["ALL", "AML"]):
            y = vi_tri + (i - 0.5) * cao
            ax.barh(y, dem[lop], height=cao, color=PALETTE[lop],
                    hatch="//" if lop == "AML" else "", edgecolor="white", label=lop)
            # (E) Ghi số ngay cuối thanh
            for yy, so in zip(y, dem[lop]):
                if so > 0:
                    ax.text(so + 0.4, yy, str(so), va="center", fontsize=co_chu)

        # (F) Trang trí từng panel
        ax.set_title(f"Tập {tap} (n = {len(con)})", fontsize=co_chu + 2)
        ax.set_xlabel("Số mẫu", fontsize=co_chu)
        ax.tick_params(labelsize=co_chu)
        ax.spines[["top", "right"]].set_visible(False)

    # (G) Phần chung cho cả hình
    axes[0].set_yticks(vi_tri, nhan_truc_y)
    axes[0].invert_yaxis()
    axes[0].set_xlim(0, 30)
    axes[1].legend(title="Lớp", fontsize=co_chu, title_fontsize=co_chu, frameon=False)
    fig.suptitle("F1. Thiết kế mẫu: số mẫu theo nguồn × loại mô × lớp", fontsize=co_chu + 3)
    fig.tight_layout()
    return fig

In [ ]:
#ban bao cao
fig = ve_F1(samples)
fig.savefig(REPORT_FIG_DIR / "F01_thiet_ke_mau_report.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#ban slide
fig = ve_F1(samples, co_chu=16, kich_thuoc=(14, 7))
fig.savefig(SLIDE_FIG_DIR / "F01_thiet_ke_mau_slide.png", dpi=150, bbox_inches="tight")
plt.show()

**Nhận xét:** Trong tập train, mỗi lớp chỉ có đúng một thanh: toàn bộ ALL đến từ DFCI–BM và toàn bộ AML đến từ CALGB–BM. Hình cho thấy trực quan việc lớp bệnh bị gây nhiễu hoàn toàn với nguồn mẫu: sự khác biệt giữa ALL và AML có thể đến từ bệnh, nhưng cũng có thể đến từ nguồn mẫu. Tập test đa dạng hơn về cả nguồn (thêm CCG và St-Jude) lẫn loại mô (có 10 mẫu PB, trong khi train hoàn toàn không có), nên kết quả trên test cho biết mô hình có tổng quát hóa được sang nguồn và loại mô chưa từng gặp khi huấn luyện hay không. Hàng St-Jude–BM là hàng duy nhất có cả ALL và AML, nên đóng vai trò quan trọng trong việc kiểm tra mô hình học "bệnh" hay học "nguồn mẫu".

## 3. Confounding: lớp bệnh, tuổi và nguồn mẫu
Dữ liệu không có cột tuổi bệnh nhân; bài báo Golub et al. (1999) ghi rõ nhóm tuổi theo từng nguồn mẫu (chú thích 13 và 23).

Chú thích 13 (tập train): "The 27 ALL samples were derived from childhood ALL patients treated on Dana-Farber Cancer Institute (DFCI) protocols… The 11 adult AML samples were similarly obtained from the Cancer and Leukemia Group B (CALGB) leukemia cell bank."

Chú thích 23 (tập test): "The ALL samples were obtained from the DFCI childhood ALL bank (n = 17) or St. Jude Children's Research Hospital (SJCRH) (n = 3)… the AML samples in the independent data set were derived from both adults and children… CALGB (adult AML, n = 4), SJCRH (childhood AML, n = 5), or the Children's Cancer Group (childhood AML, n = 5)… SJCRH… used a very different protocol (hypotonic lysis… Qiagen)."

In [ ]:
# Nhóm tuổi theo nguồn mẫu, căn cứ chú thích 13 và 23 của bài báo
NHOM_TUOI = {"DFCI": "trẻ em", "St-Jude": "trẻ em", "CCG": "trẻ em", "CALGB": "người lớn"}

nhom_tuoi = samples["source"].map(NHOM_TUOI).rename("nhom_tuoi")
assert nhom_tuoi.notna().all(), "Có nguồn mẫu chưa được gán nhóm tuổi"

pd.crosstab([samples["split"], samples["class"]], nhom_tuoi,
            margins=True, margins_name="Tổng")

**Phân tích confounding:**
1. Train: ba yếu tố trùng khớp hoàn toàn với nhau: lớp bệnh, nhóm tuổi và nguồn mẫu – ALL = trẻ em = DFCI, AML = người lớn = CALGB (chú thích 13). Vì vậy một gen "phân biệt ALL/AML" tìm được trên tập train có thể thực chất phản ánh tuổi hoặc nguồn mẫu thay vì bệnh; dữ liệu train không đủ để tách ba hiệu ứng này. (Loại mô không gây nhiễu vì toàn bộ train là BM.)
2. ALL: cả bộ dữ liệu không có mẫu ALL người lớn nào (47/47 là trẻ em, chú thích 13 và 23). Do đó không kiểm tra được mô hình có nhận đúng ALL ở người lớn hay không; kết luận của dự án chỉ áp dụng cho ALL trẻ em.
3. Test – AML: có 10 mẫu AML trẻ em, từ St-Jude (5) và CCG (5) (chú thích 23), trong khi tập train không có AML trẻ em nào. Nếu mô hình học nhầm "trẻ em → ALL", nó sẽ đoán nhầm các mẫu này thành ALL. Vì vậy nhóm mẫu này là phép kiểm tra quan trọng nhất cho việc mô hình phân biệt bệnh thật hay chỉ dựa vào tuổi.
4. Quy trình phòng lab: bài báo ghi nhận prediction strength trung bình thấp hơn ở mẫu của phòng lab dùng quy trình khác (St-Jude). Điều này phù hợp với tín hiệu yếu ở PC3 khi cố định bệnh: trong AML, St-Jude so với phần còn lại có p = 0,017 và theo nguồn có p = 0,032, nhưng đều mất ý nghĩa sau hiệu chỉnh BH. Đây chỉ là tín hiệu yếu về khác biệt do quy trình, chưa đủ để kết luận.
5. Đề xuất cho giai đoạn 2:
   - Khi báo cáo kết quả trên tập test, tách riêng ba nhóm: AML trẻ em (St-Jude + CCG, n = 10), mẫu St-Jude (n = 8) và mẫu PB (n = 10) – TV2 (khung đánh giá 2.4), TV5 (phân tích lỗi 2.5).
   - Tái lập prediction strength của Golub và so sánh PS theo nguồn mẫu, kiểm tra lại nhận xét "PS thấp hơn ở St-Jude" – TV1 (2.1).
   - Với danh sách gen cuối, so sánh biểu hiện giữa AML trẻ em và AML người lớn trong tập test: gen nào khác biệt mạnh theo tuổi cần được đánh dấu là "có thể phản ánh tuổi" khi diễn giải sinh học – TV4 (2.2), TV5 (2.5).
   - Ghi rõ giới hạn "không có ALL người lớn" trong phần kết luận của báo cáo.

## 4. Batch effect: PCA theo nguồn mẫu và loại mô

PCA trên 3571 gen đã lọc (log10, chuẩn hóa z-score). Màu = nguồn mẫu hoặc loại mô; hình dạng = lớp (ALL tròn, AML tam giác).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from src.config import RANDOM_SEED

# (A) Đọc ma trận đã lọc gen của TV3
Xf = pd.read_parquet(CURATED_DIR / "X_log10_filtered.parquet")
Xf.index = Xf.index.astype(int)
assert Xf.index.equals(samples.index), "Xf và samples lệch thứ tự mẫu"
print("Xf:", Xf.shape)

# (B) Chuẩn hóa mỗi gen về trung bình 0, độ lệch chuẩn 1
Z = StandardScaler().fit_transform(Xf)

# (C) PCA giữ 10 thành phần đầu
pca = PCA(n_components=10, random_state=RANDOM_SEED)
diem = pca.fit_transform(Z)
ty_le = pca.explained_variance_ratio_

# (D) Gom tọa độ PC1–PC3 với thông tin mẫu vào một bảng
pcs = pd.DataFrame(diem[:, :3], index=Xf.index, columns=["PC1", "PC2", "PC3"])
pcs = pcs.join(samples[["split", "class", "source", "tissue"]])

print("Tỉ lệ phương sai PC1–PC3 (%):", (ty_le[:3] * 100).round(1))
pcs.head()

In [ ]:
MAU_NGUON = {"DFCI": "#009E73", "CALGB": "#CC79A7", "CCG": "#56B4E9", "St-Jude": "#D55E00"}
MAU_MO = {"BM": "#7F7F7F", "PB": "#D55E00"}

def ve_pca_batch(pcs, ty_le, co_chu=10, kich_thuoc=(11, 5)):
    """PCA PC1–PC2: panel trái tô màu theo nguồn, panel phải theo loại mô; hình dạng = lớp."""
    fig, axes = plt.subplots(1, 2, figsize=kich_thuoc, sharex=True, sharey=True)
    cau_hinh = [("source", MAU_NGUON, "Nguồn mẫu"), ("tissue", MAU_MO, "Loại mô")]

    for ax, (cot, bang_mau, tieu_de) in zip(axes, cau_hinh):
        # (A) Vẽ điểm: mỗi tổ hợp (nhóm, lớp) một lần, màu theo nhóm, hình dạng theo lớp
        for nhom, mau in bang_mau.items():
            for lop, dau in MARKERS.items():
                con = pcs[(pcs[cot] == nhom) & (pcs["class"] == lop)]
                ax.scatter(con["PC1"], con["PC2"], c=mau, marker=dau, s=50,
                           edgecolors="black", linewidths=0.5, alpha=0.85)

        # (B) Chú giải: tách riêng "màu = nhóm" và "hình = lớp"
        for nhom, mau in bang_mau.items():
            ax.scatter([], [], c=mau, marker="s", s=60, label=nhom)
        for lop, dau in MARKERS.items():
            ax.scatter([], [], facecolors="white", edgecolors="black", marker=dau, s=50, label=lop)
        ax.legend(title=tieu_de, fontsize=co_chu - 1, title_fontsize=co_chu, frameon=False,
                  loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3)
        # (C) Trang trí
        ax.set_title(f"Tô màu theo {tieu_de.lower()}", fontsize=co_chu + 2)
        ax.set_xlabel(f"PC1 ({ty_le[0]:.1%} phương sai)", fontsize=co_chu)
        ax.tick_params(labelsize=co_chu)
        ax.spines[["top", "right"]].set_visible(False)

    axes[0].set_ylabel(f"PC2 ({ty_le[1]:.1%} phương sai)", fontsize=co_chu)
    fig.suptitle("PCA 72 mẫu (3571 gen đã lọc, z-score): kiểm tra batch effect", fontsize=co_chu + 3)
    fig.tight_layout()
    return fig

In [ ]:
fig = ve_pca_batch(pcs, ty_le)
fig.savefig(REPORT_FIG_DIR / "F04d_pca_batch_report.png", dpi=300, bbox_inches="tight")
plt.show()

fig = ve_pca_batch(pcs, ty_le, co_chu=16, kich_thuoc=(15, 7))
fig.savefig(SLIDE_FIG_DIR / "F04d_pca_batch_slide.png", dpi=150, bbox_inches="tight")
plt.close(fig)

**Nhận xét:** ALL và AML tách nhau chủ yếu theo PC2 (ALL phía trên, AML phía dưới), còn PC1 – trục giữ nhiều phương sai nhất (16,1%) – không tách hai lớp. Trong cùng một lớp, các nguồn nằm lẫn vào nhau: tam giác CALGB xen kẽ với tam giác AML của CCG và St-Jude; điểm tròn St-Jude nằm giữa các điểm tròn DFCI. Tương tự, mẫu PB nằm lẫn với mẫu BM cùng lớp. Như vậy, bằng mắt, cấu trúc chính của dữ liệu theo bệnh, không thấy dấu hiệu rõ của batch effect theo nguồn hay loại mô. Nhận định này cần được kiểm chứng bằng kiểm định thống kê trên PC1–PC3 trong từng lớp (phần tiếp theo), đặc biệt vì PC1 chưa được giải thích và số mẫu ở CCG (5) và St-Jude (8) còn nhỏ.

### 4.2. Kiểm định batch effect trên PC1–PC3

Kruskal–Wallis so sánh giá trị PC giữa các nhóm. Vì nguồn mẫu bị gây nhiễu với lớp bệnh, kiểm định trên toàn bộ 72 mẫu sẽ lẫn hiệu ứng bệnh; kiểm định đúng phải làm **trong từng lớp**.

In [ ]:
from scipy.stats import kruskal

PCS = ["PC1", "PC2", "PC3"]
print("Kiểm định batch effect (Kruskal–Wallis theo source, cả 72 mẫu):")
for pc in PCS:
    p_nguon = kruskal(*[g for _, g in pcs[pc].groupby(pcs["source"])]).pvalue
    p_lop = kruskal(*[g for _, g in pcs[pc].groupby(pcs["class"])]).pvalue
    print(f"  {pc}: p (theo source) = {p_nguon:.2g}   |   p (theo class) = {p_lop:.2g}")

In [ ]:
from statsmodels.stats.multitest import multipletests

ket_qua = []
for lop in ["ALL", "AML"]:
    # (A) Chỉ giữ mẫu của một lớp -> bệnh cố định, khác biệt còn lại mới có thể là batch
    con = pcs[pcs["class"] == lop]

    # (B) Ba yếu tố cần kiểm tra: nguồn, loại mô, St-Jude so với phần còn lại
    yeu_to = {
        "source": con["source"],
        "tissue": con["tissue"],
        "St-Jude vs còn lại": con["source"].where(con["source"] == "St-Jude", "khác"),
    }
    if lop == "ALL":
        del yeu_to["St-Jude vs còn lại"]   # ALL chỉ có DFCI và St-Jude -> trùng với "source"

    # (C) Chạy Kruskal–Wallis cho từng yếu tố × từng PC
    for ten, nhan in yeu_to.items():
        co_mau = " / ".join(f"{k} {v}" for k, v in nhan.value_counts().sort_index().items())
        for pc in PCS:
            nhom = [g for _, g in con[pc].groupby(nhan)]
            H, p = kruskal(*nhom)
            ket_qua.append({"lop": lop, "yeu_to": ten, "co_mau": co_mau,
                            "PC": pc, "H": H, "p": p})

# (D) Gom thành bảng, hiệu chỉnh Benjamini–Hochberg, lưu file
bang = pd.DataFrame(ket_qua)
bang["p_BH"] = multipletests(bang["p"], method="fdr_bh")[1]
bang["co_y_nghia"] = bang["p_BH"] < 0.05
bang.to_csv(METRICS_DIR / "batch_effect_tests.csv", index=False)
bang.round(4)

**Kết luận H6:**
- Kiểm định gộp 72 mẫu: p theo nguồn rất nhỏ ở PC2 (7,7e-07) và PC3 (9,8e-06), nhưng p theo lớp bệnh cũng rất nhỏ ở đúng các PC này. Vì nguồn mẫu gần như trùng với lớp bệnh (DFCI chỉ có ALL; CALGB, CCG chỉ có AML), kiểm định gộp không tách được hiệu ứng nguồn khỏi hiệu ứng bệnh nên không dùng để kết luận.
- Kiểm định trong từng lớp: khi đã cố định lớp bệnh, sau hiệu chỉnh Benjamini–Hochberg không có kiểm định nào trong 15 kiểm định có ý nghĩa (p_BH ≥ 0,08). Nghĩa là trong cùng một bệnh, nguồn mẫu và loại mô không làm thay đổi rõ rệt vị trí của mẫu trên PC1–PC3. Khác biệt lớn ở kiểm định gộp vì vậy chủ yếu do bệnh.
- Tín hiệu yếu: ba kiểm định có p gốc < 0,05 nhưng mất ý nghĩa sau hiệu chỉnh BH: BM vs PB trong ALL (p = 0,0055), nguồn trong AML (p = 0,032) và St-Jude vs còn lại trong AML (p = 0,017). Cả ba đều ở PC3. Đây chỉ là tín hiệu, không đủ bằng chứng khẳng định có khác biệt do nguồn hoặc loại mô.
- Giới hạn: cỡ mẫu nhỏ (St-Jude ALL chỉ có 3, CCG 5, PB của AML 4). "Không có ý nghĩa" không đồng nghĩa với "chắc chắn không có batch effect", vì kiểm định có thể không đủ mạnh để phát hiện. Ngoài ra, có một cặp không thể kiểm định được dù dùng cách nào: DFCI so với CALGB trong cùng một lớp, vì DFCI chỉ có ALL còn CALGB chỉ có AML. Do đó, trong tập train, nếu có batch effect giữa hai nguồn này thì nó hoàn toàn trộn lẫn với khác biệt ALL/AML và dữ liệu không thể loại trừ khả năng đó.
- Kết luận: Không có dấu hiệu batch effect rõ ràng theo nguồn mẫu hay loại mô trên PC1–PC3; cấu trúc chính của dữ liệu (PC2, PC3) chủ yếu phản ánh lớp bệnh, chỉ có tín hiệu yếu ở PC3. Tuy nhiên, do cỡ mẫu nhỏ và nguồn bị gây nhiễu hoàn toàn với lớp trong tập train, không có dấu hiệu rõ ràng không đồng nghĩa với không có batch effect. Vì vậy, kết quả trên tập test (đặc biệt các mẫu St-Jude, nguồn duy nhất có cả hai lớp) là phép kiểm tra quan trọng cho việc mô hình học bệnh hay học nguồn.


## 5. Cấu hình thí nghiệm giai thừa (`config/experiment_config.yaml`)

6 phương pháp chọn gen × 6 giá trị k × 7 bộ phân loại = 252 cấu hình; `tests/test_experiment_config.py` kiểm tra các ràng buộc.

In [ ]:
import yaml
from src.config import CONFIG_DIR

with open(CONFIG_DIR / "experiment_config.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

so_cau_hinh = len(cfg["selection"]) * len(cfg["k"]) * len(cfg["classifiers"])
print("Số cấu hình:", so_cau_hinh)
print("outer_repeats =", cfg["cv"]["outer_repeats"], "| số seed =", len(cfg["seeds"]))
cfg